# Module 03 — ML for Forecasting

## Notebook 1 · Forecasting Fundamentals: Patterns, Losses, and Metrics

**Goals**
1. Audit a panel forecasting dataset and split it correctly along the time axis.
2. Classify each forecast key as *Smooth*, *Erratic*, *Intermittent*, or *Lumpy* using the **Syntetos–Boylan** quadrant (ADI vs CV²).
3. Pick the right **loss function** — MSE, MAE, **Poisson**, **Tweedie**, or Quantile — for each demand pattern, and explain *why*.
4. Pick the right **evaluation metric** — RMSE, WAPE, MASE, RMSSE — and avoid the classic MAPE-on-zero-demand trap.

**How this notebook fits in.** Every later notebook (LightGBM, Prophet, tuning, explainability) refers back to the loss-function and metric choices we make here.


In [22]:
# === Colab / local setup ====================================================
# 1. Install dependencies (uncomment the pip line on first Colab run).
# !pip install -q lightgbm==4.* prophet plotly optuna shap pandas numpy scikit-learn pyarrow

# 2. Make the `utils` package importable. Two options:
#    (a) Notebook is sitting next to a `utils/` folder (recommended).
#    (b) The package is uploaded as a zip; unzip it and `sys.path.append(...)`.
import os, sys
HERE = os.path.dirname(os.path.abspath("__file__"))  # may be empty in Colab
for cand in [".", "..", "/content", "/content/ml_forecasting_tutorial"]:
    if os.path.isdir(os.path.join(cand, "utils")):
        sys.path.insert(0, cand)
        break

# 3. Standard imports for every notebook.
import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = "colab"   # works in Colab + Jupyter

# 4. Tell pandas to display nicely.
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


In [23]:
# === Dataset configuration ==================================================
# EDIT THIS CELL TO POINT AT YOUR DATASET
DATA_PATH    = "./dataset/m5/m5_small.csv"             # path to your CSV / parquet
DATE_COL     = "date"                              # date column
TARGET_COL   = "sales"                          # target / forecast column
KEY_COLS     = ['item_id',
                'dept_id',
                'cat_id',
                'store_id',
                'state_id']                           # columns identifying a unique series
FREQ         = "D"                              # 'D'=daily, 'W'=weekly, 'M'=monthly
HOLDOUT_DAYS = 28                               # length of test horizon


### 1. Load the dataset

`utils.data_utils.load_forecasting_data` is a thin wrapper around `pd.read_csv` /
`pd.read_parquet` that:

* parses the date column,
* validates that the target and key columns exist,
* sorts by `(*key_cols, date_col)` — required for any per-key lag / rolling
  feature engineering downstream.

> **Note.** The function works for *any* panel forecasting dataset by passing the
> right column names — there is no hard-coding to a specific Kaggle competition.


In [24]:
from utils.data_utils import load_forecasting_data

df = load_forecasting_data(
    path=DATA_PATH,
    date_col=DATE_COL,
    target_col=TARGET_COL,
    key_cols=KEY_COLS
)
print(f"shape         : {df.shape}")
print(f"date range    : {df[DATE_COL].min().date()}  →  {df[DATE_COL].max().date()}")
print(f"forecast keys : {df[KEY_COLS].drop_duplicates().shape[0]:,}")
df.head()


shape         : (8582723, 53)
date range    : 2015-03-08  →  2016-06-19
forecast keys : 18,294


,id,date,item_id,dept_id,cat_id,store_id,state_id,d,sales,sell_price,snap_CA,snap_TX,snap_WI,weight,event_name_1_Chanukah End,event_name_1_Christmas,event_name_1_Cinco De Mayo,event_name_1_ColumbusDay,event_name_1_Easter,event_name_1_Eid al-Fitr,event_name_1_EidAlAdha,event_name_1_Father's day,event_name_1_Halloween,event_name_1_IndependenceDay,event_name_1_LaborDay,...,event_name_1_MemorialDay,event_name_1_Mother's day,event_name_1_NBAFinalsEnd,event_name_1_NBAFinalsStart,event_name_1_NewYear,event_name_1_OrthodoxChristmas,event_name_1_OrthodoxEaster,event_name_1_Pesach End,event_name_1_PresidentsDay,event_name_1_Purim End,event_name_1_Ramadan starts,event_name_1_StPatricksDay,event_name_1_SuperBowl,event_name_1_Thanksgiving,event_name_1_ValentinesDay,event_name_1_VeteransDay,event_name_2_Cinco De Mayo,event_name_2_Father's day,event_name_2_OrthodoxEaster,event_type_1_Cultural,event_type_1_National,event_type_1_Religious,event_type_1_Sporting,event_type_2_Cultural,event_type_2_Religious
0,FOODS_1_001_TX_1_evaluation,2015-03-08,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,1500,1.0,2.24,1,0,1,0.000015,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,FOODS_1_001_TX_1_evaluation,2015-03-09,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,1501,1.0,2.24,1,1,1,0.000015,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,FOODS_1_001_TX_1_evaluation,2015-03-10,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,1502,0.0,2.24,1,0,0,0.000015,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,FOODS_1_001_TX_1_evaluation,2015-03-11,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,1503,0.0,2.24,0,1,1,0.000015,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,FOODS_1_001_TX_1_evaluation,2015-03-12,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,1504,0.0,2.24,0,1,1,0.000015,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


### 2. Audit the panel

`validate_panel` reports per-key gaps in the date index, zero / negative observations, and missing targets. Any **gap** in the per-key date index means a downstream `shift(1)` will jump over a calendar day, silently corrupting your lag features.

If you see gaps, run `complete_panel(...)` to insert zero-demand rows for the missing dates.


In [25]:
from utils.data_utils import validate_panel, complete_panel

panel_summary = validate_panel(df, DATE_COL, KEY_COLS, TARGET_COL, freq=FREQ, verbose=True)
panel_summary.head()


Panel summary: 18294 forecast keys, freq='D'
  Total missing date rows : 0
  Total zero observations : 4,946,320
  Total negatives         : 0
  Total NaNs in target    : 512,232


,item_id,dept_id,cat_id,store_id,state_id,n_obs,first_date,last_date,missing_dates,n_zeros,n_negatives,n_nans
0,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,470,2015-03-08,2016-06-19,0,301,0,28
1,FOODS_1_001,FOODS_1,FOODS,TX_2,TX,470,2015-03-08,2016-06-19,0,317,0,28
2,FOODS_1_001,FOODS_1,FOODS,TX_3,TX,470,2015-03-08,2016-06-19,0,305,0,28
3,FOODS_1_001,FOODS_1,FOODS,WI_1,WI,470,2015-03-08,2016-06-19,0,310,0,28
4,FOODS_1_001,FOODS_1,FOODS,WI_2,WI,470,2015-03-08,2016-06-19,0,313,0,28


In [26]:
# Optional: only run this if `panel_summary["missing_dates"].sum() > 0`.
if panel_summary["missing_dates"].sum() > 0:
    df = complete_panel(df, DATE_COL, KEY_COLS, TARGET_COL, freq=FREQ, fill_value=0.0)
    print(f"Panel completed, new shape: {df.shape}")


### 3. Visualise a sample of forecast keys

We will not look at all keys at once (impractical), but ~8 randomly chosen ones is enough to:

* recognise the seasonality pattern,
* spot whether demand is continuous or punctuated by zeros,
* gut-check whether weekly / yearly seasonality is dominant.


In [27]:
from utils.viz import plot_multi_series

fig = plot_multi_series(df, DATE_COL, TARGET_COL, KEY_COLS, n_keys=8, title="Sample of forecast keys")
fig.show()


### 4. Classify each forecast key — Syntetos–Boylan

Two scalars summarise a series for forecasting purposes:

| Statistic | Formula | Intuition |
|-----------|--------|-----------|
| **ADI** (Average Demand Interval) | $$\text{periods} / \text{non-zero periods}$$ | how *often* demand happens (≈1 for smooth, large for sparse) |
| **CV²** (squared coefficient of variation of non-zero sizes) | $$(\sigma / \mu)^2$$ on non-zero observations | how *variable* the demand size is when it does happen |

The Syntetos–Boylan quadrant uses thresholds 1.32 (ADI) and 0.49 (CV²):

* **Smooth** — frequent, predictable. Vanilla MSE / RMSE works well.
* **Erratic** — frequent but variable size. Try Huber or MAE.
* **Intermittent** — rare but predictable size. **Tweedie** (variance power closer to 1, i.e. Poisson-ish) shines.
* **Lumpy** — rare *and* variable. The hardest case. **Tweedie** with power ≈ 1.3–1.5 is the industry default.

We are about to train one global LightGBM model across all keys. Knowing which patterns dominate guides our objective choice.


In [28]:
from utils.data_utils import classify_demand
from utils.viz import plot_demand_classification

classification = classify_demand(df, DATE_COL, KEY_COLS, TARGET_COL)
print(classification["pattern"].value_counts())
classification.head()


pattern
Intermittent    14586
Lumpy            2268
Smooth           1190
Erratic           250
Name: count, dtype: int64


,item_id,dept_id,cat_id,store_id,state_id,ADI,CV2,pattern,mean_demand,zero_share
0,FOODS_1_001,FOODS_1,FOODS,TX_1,TX,3.333333,2.061754,Lumpy,NaN,0.640426
1,FOODS_1_001,FOODS_1,FOODS,TX_2,TX,3.760000,0.585366,Lumpy,NaN,0.674468
2,FOODS_1_001,FOODS_1,FOODS,TX_3,TX,3.430657,0.385974,Intermittent,NaN,0.648936
3,FOODS_1_001,FOODS_1,FOODS,WI_1,WI,3.560606,0.370849,Intermittent,NaN,0.659574
4,FOODS_1_001,FOODS_1,FOODS,WI_2,WI,3.643411,0.357632,Intermittent,NaN,0.665957


In [29]:
fig = plot_demand_classification(classification)
fig.show()


### 5. Loss functions — visual intuition

The choice of **training loss** is the single biggest decision in ML forecasting. Below we plot how four standard losses penalise predictions for a fixed actual value `y_true=5`:

* **MSE** — symmetric, quadratic. Massive penalty for over- or under-prediction.
* **MAE** — symmetric, linear. Robust to outliers; minimises the *median*.
* **Poisson NLL** — asymmetric, well-defined for `y >= 0`. Small predictions are punished much more than for MSE because the log term explodes near zero.
* **Tweedie deviance** (variance power `p=1.5`) — same flavour as Poisson but with an extra dispersion parameter. The work-horse for retail demand.

Notice how Tweedie's curve is **steeper for under-prediction** than MSE — exactly what you want when you have lots of zero-demand periods that punish models with high "average" predictions.


In [30]:
from utils.viz import plot_loss_shapes
fig = plot_loss_shapes(y_true=5.0)
fig.show()


For an even sharper contrast, compare what happens when `y_true=0` (a typical intermittent day): MSE just rewards predicting zero, but Poisson and Tweedie *strongly* prefer small positive predictions — which is what makes them useful when demand is genuinely zero on most days but you still want to capture the underlying expected demand rate.


In [31]:
fig = plot_loss_shapes(y_true=0.5, title="Loss curves at y_true=0.5 (small but non-zero demand)")
fig.show()


### 6. Loss-function catalog

A reference table of every loss the rest of this tutorial will touch — its LightGBM objective name, when to use it, and what to watch out for.


In [32]:
from utils.losses import LOSS_CATALOG, recommend_loss
import pandas as pd
pd.set_option("display.max_colwidth",1000)

cat_df = pd.DataFrame(LOSS_CATALOG).T.reset_index().rename(columns={"index": "key"})
cat_df


,key,name,lightgbm_objective,domain,use_when,watch_out
0,regression_l2,MSE / RMSE,regression,y in R,"Smooth / continuous demand. Symmetric, penalises large errors quadratically.",Very sensitive to outliers; can drive predictions far from zero on lumpy data.
1,regression_l1,MAE,regression_l1,y in R,"Robust regression; minimises the median, not the mean. Good when you have heavy-tailed errors.",Gradient is sign(error) — slower convergence than MSE in the central region.
2,huber,Huber,huber,y in R,"Compromise between MSE and MAE; quadratic near zero, linear beyond delta.",Requires tuning the alpha (huber delta) parameter.
3,poisson,Poisson,poisson,"y in {0, 1, 2, ...}",Count data with mean ~ variance. Predictions are positive by construction (model outputs log-mean).,Overdispersion (variance > mean) is common in retail; switch to Tweedie or Negative Binomial.
4,tweedie,Tweedie,tweedie,y >= 0,"Zero-inflated, right-skewed demand (intermittent / lumpy). Tweedie is *the* default for retail forecasting at scale (Walmart M5 winners, Amazon DeepAR alternatives).","Tune `tweedie_variance_power` in (1, 2). 1.0 -> Poisson, 2.0 -> Gamma. Typical retail values: 1.1 - 1.5."
5,quantile,Quantile (Pinball),quantile,y in R,"You want a *probabilistic* forecast. Train one model per quantile (e.g. 0.1, 0.5, 0.9) to get prediction intervals.",Quantile crossing can occur (q=0.9 prediction below q=0.5); post-process by sorting if needed.


In [33]:
# Recommend an objective for each pattern present in the data
for pat in classification["pattern"].unique():
    print(f"  {pat:>14} -> objective = '{recommend_loss(pat)}'")


           Lumpy -> objective = 'tweedie'
    Intermittent -> objective = 'tweedie'
          Smooth -> objective = 'regression'
         Erratic -> objective = 'huber'


### 7. Evaluation metrics

Training loss and evaluation metric do **not** have to match. You can train with Tweedie (because that yields the best point predictions for skewed data) and evaluate with WAPE (because business stakeholders understand "% error" intuitively).

The table below summarises which metric to report for each demand pattern:

| Pattern | Recommended report metrics | Avoid |
|---|---|---|
| Smooth | RMSE, MAE, MAPE | — |
| Erratic | RMSE, MAE, WAPE | — |
| Intermittent | **WAPE**, MAE, RMSE, Bias | MAPE (undefined on zero) |
| Lumpy | **WAPE**, MAE, Tweedie deviance | MAPE |
| Cross-key aggregation | WAPE, **MASE**, **RMSSE** | RMSE (scale-dependent) |

A baseline forecast — e.g., "predict last week's value" — gives us something to scale against. We compute MASE / RMSSE relative to seasonal-naïve in-sample errors, so a model with MASE=0.5 is twice as accurate as the seasonal-naïve baseline.


### 8. Train / test split

Forecasting splits must be along **time**, never random. We hold out the last `HOLDOUT_DAYS` calendar days so the test set is the *future* of every key.


In [35]:
from utils.data_utils import time_based_split, summarize_split

cutoff = df[DATE_COL].max() - pd.Timedelta(days=HOLDOUT_DAYS - 1)
train_df, test_df = time_based_split(df, DATE_COL, cutoff=cutoff)
summarize_split(train_df, test_df, DATE_COL)


  train: 8,070,491 rows  |  2015-03-08 → 2016-05-22
  test :  512,232 rows  |  2016-05-23 → 2016-06-19


### Recap

* **Audit** the panel: gaps in dates corrupt lag features.
* **Classify** demand patterns: ADI/CV² → Smooth / Erratic / Intermittent / Lumpy.
* **Pick the loss** that matches the pattern: Tweedie for retail-style data is the safe default.
* **Pick the metric** carefully: avoid MAPE on intermittent demand; prefer WAPE / MASE.
* **Split by time**, hold out the last *H* days as the test set.

In the next notebook we build the **causal feature set** — lags, rolling stats, calendar — that will feed every LightGBM model in this tutorial.
